# PromptGFM-Bio — Kaggle Training Notebook
**Gene-Phenotype Prediction for Rare Diseases**

⚙️ Before running:
- Go to **Settings → Accelerator → GPU T4 x2** (or GPU P100)
- Turn on **Internet** in Settings
- *(Optional)* Add your project GitHub repo URL in the cell below

**Session time limit**: 9 h — checkpoints are saved after every epoch, so you can resume safely.

## 1. Environment Check

In [5]:
import sys, subprocess, torch, os

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    print(f'GPU count: {torch.cuda.device_count()}')
else:
    print('⚠️  No GPU detected! Enable GPU in Notebook Settings.') 

Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch : 2.9.0+cu126
CUDA    : 12.6
GPU     : Tesla T4
VRAM    : 15.6 GB
GPU count: 2


## 2. Install PyTorch Geometric

In [6]:
# Detect exact torch + CUDA version to pick the right wheel URL
import torch

TORCH_VER = torch.__version__.split('+')[0]        # e.g. '2.1.0'
CUDA_VER  = torch.version.cuda.replace('.', '')    # e.g. '121'
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'Installing PyG wheels from: {WHEEL_URL}')

packages = [
    'torch-scatter',
    'torch-sparse',
    'torch-cluster',
    'torch-spline-conv',
    'torch-geometric',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '-f', WHEEL_URL] + packages,
    check=True
)
print('✅ PyTorch Geometric installed')

Installing PyG wheels from: https://data.pyg.org/whl/torch-2.9.0+cu126.html
✅ PyTorch Geometric installed


In [7]:
# Install remaining dependencies
extra = [
    'transformers>=4.40.0',
    'sentence-transformers>=2.7.0',
    'biopython>=1.84',
    'networkx>=3.2',
    'wandb>=0.17.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + extra, check=True)
print('✅ Extra packages installed')

✅ Extra packages installed


## 3. Clone Project Code from GitHub

In [8]:
# ── Edit this URL to point to your GitHub repo ──────────────────────────────
GITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = '/kaggle/working/PromptGFM-Bio'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, PROJECT_DIR], check=True)
    print(f'✅ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)
    print(f'✅ Pulled latest changes in {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

Already up to date.
✅ Pulled latest changes in /kaggle/working/PromptGFM-Bio
Working directory: /kaggle/working/PromptGFM-Bio


## 4. Create Directory Structure

In [9]:
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/splits',
    'checkpoints/promptgfm_film',
    'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Directories created')

✅ Directories created


## 5. Download Biomedical Datasets
Total download: ~1.5 GB (BioGRID + STRING + DisGeNET + HPO).  
This step takes a few minutes. Skip it if you have already preprocessed data stored as a Kaggle Dataset.

In [10]:
result = subprocess.run(
    [sys.executable, 'scripts/download_data.py', '--dataset', 'all'],
    capture_output=False   # show live output
)
print('Download exit code:', result.returncode)

INFO:src.data.download:======================================================================
INFO:src.data.download:Starting download of all biomedical datasets...
INFO:src.data.download:This may take 30-60 minutes depending on your connection
INFO:src.data.download:Total size: ~1.5 GB
INFO:src.data.download:======================================================================
INFO:src.data.download:
[1/4] BioGRID Protein-Protein Interactions
INFO:src.data.download:Downloading BioGRID database (~500MB)...
INFO:src.data.download:Downloading from https://downloads.thebiogrid.org/Download/BioGRID/Release-Archive/BIOGRID-4.4.224/BIOGRID-ALL-4.4.224.tab3.zip (attempt 1/3)



PromptGFM-Bio Data Download

Dataset: all
Force re-download: False
Skip failing: True



BIOGRID-ALL-4.4.224.tab3.zip: 160MB [00:02, 70.1MB/s] 
INFO:src.data.download:Successfully downloaded to /kaggle/working/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.zip
INFO:src.data.download:File BIOGRID-ALL-4.4.224.tab3.zip md5: ed6819e6e65ec7e685e85a0d66060ba9
INFO:src.data.download:Extracting BIOGRID-ALL-4.4.224.tab3.zip...
INFO:src.data.download:Extraction complete!
INFO:src.data.download:
[2/4] STRING Protein Network
INFO:src.data.download:Downloading STRING database for organism 9606 (~700MB)...
INFO:src.data.download:Downloading from https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz (attempt 1/3)
9606.protein.links.v12.0.txt.gz: 100%|██████████| 83.2M/83.2M [00:04<00:00, 17.5MB/s]
INFO:src.data.download:Successfully downloaded to /kaggle/working/PromptGFM-Bio/data/raw/string/9606.protein.links.v12.0.txt.gz
INFO:src.data.download:File 9606.protein.links.v12.0.txt.gz md5: 5b7c91d02926e723ad30e81ab9d009a4
INFO:src.data.download


✓ Successfully downloaded 4/4 datasets

✓ DOWNLOAD COMPLETE

Next steps:
  1. Run preprocessing: python scripts/preprocess_all.py
  2. Check downloaded files in: data/raw/

Download exit code: 0


INFO:src.data.download:File phenotype.hpoa md5: b249dc767256e904d1e92895184009b2
INFO:src.data.download:
INFO:src.data.download:Download Summary:
INFO:src.data.download:✓ BIOGRID: 2 files downloaded
INFO:src.data.download:✓ STRING: 3 files downloaded
INFO:src.data.download:✓ DISGENET: 2 files downloaded
INFO:src.data.download:✓ HPO: 4 files downloaded
INFO:src.data.download:======================================================================


## 6. Preprocess Data (Build Knowledge Graph)

In [11]:
result = subprocess.run(
    [sys.executable, 'scripts/preprocess_all.py'],
    capture_output=False
)
print('Preprocessing exit code:', result.returncode)

# Confirm the graph was built
graph_path = Path('data/processed/biomedical_graph.pt')
if graph_path.exists():
    size_mb = graph_path.stat().st_size / 1e6
    print(f'✅ Graph ready: {graph_path}  ({size_mb:.1f} MB)')
else:
    print('⚠️  Graph file not found — check preprocessing logs above')

INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Starting enhanced preprocessing pipeline...
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Options:
INFO:src.data.preprocess:  HPO Bridge: True
INFO:src.data.preprocess:  Orphadata: True
INFO:src.data.preprocess:  UniProt: False
INFO:src.data.preprocess:  Pathways: False
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:
[Step 1] Parsing PPI networks...
INFO:src.data.preprocess:Parsing BioGRID from /kaggle/working/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.txt...
ERROR:src.data.preprocess:Failed to parse BioGRID: Usecols do not match columns, columns expected but not found: ['Organism Interactor B', 'Organism Interactor A']
INFO:src.data.preprocess:Parsing STRING from /kaggle/working/PromptGFM-Bio/data/raw/strin


PromptGFM-Bio Enhanced Preprocessing Pipeline

Configuration:
  Force reprocess: False
  HPO Bridge: True
  Orphadata: True
  UniProt: False
  Pathways: False


✓ PREPROCESSING COMPLETE

Next steps:
  1. Create dataset splits: python -m src.data.dataset
  2. Check graph file: data/processed/biomedical_graph.pt
  3. View statistics: data/processed/biomedical_graph_stats.txt

Preprocessing exit code: 0
✅ Graph ready: data/processed/biomedical_graph.pt  (313.4 MB)


## 7. (Optional) Restore Previous Checkpoint
If you have a saved checkpoint from a previous Kaggle session stored as a Kaggle Dataset, copy it here.

In [12]:
import shutil

# Path to your checkpoint Kaggle Dataset (e.g. /kaggle/input/promptgfm-checkpoints/)
CHECKPOINT_DATASET = '/kaggle/input/promptgfm-checkpoints'   # change if needed

if os.path.exists(CHECKPOINT_DATASET):
    dest = Path('checkpoints/promptgfm_film')
    for f in Path(CHECKPOINT_DATASET).glob('*'):
        shutil.copy2(f, dest / f.name)
    print(f'✅ Checkpoints restored from {CHECKPOINT_DATASET}')
else:
    print('No checkpoint dataset found — starting fresh (this is fine for first run)')

No checkpoint dataset found — starting fresh (this is fine for first run)


## 8. W&B Login (Optional)

In [13]:
# Paste your W&B API key here, or set use_wandb: false in the config
WANDB_API_KEY = ''   # leave empty to skip

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled — training metrics will be logged to stdout only')

W&B disabled — training metrics will be logged to stdout only


## 9. Train
Uses `configs/kaggle_config.yaml` which is tuned for the Kaggle T4 (16 GB VRAM).  
To **resume** from the last saved checkpoint, set `RESUME = True`.

In [ ]:
import torch as _torch
from pathlib import Path as _Path

# --- Auto-detect in-session checkpoints for same-session restart ---
_ckpt_dir = _Path("checkpoints/promptgfm_film")
_ckpts = sorted(
    _ckpt_dir.glob("checkpoint_epoch_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1])
) if _ckpt_dir.exists() else []

# RESUME_CHECKPOINTS=True  = resuming from a Kaggle Dataset (next session)
# _ckpts non-empty         = in-session restart (same kernel, stopped+restarted)
RESUME = RESUME_CHECKPOINTS or bool(_ckpts)

# Always use train.py  (resume_training.py is interactive and hangs in notebooks)
base_args = ["scripts/train.py", "--config", "configs/kaggle_config.yaml"]
if RESUME and _ckpts:
    _last_ckpt = str(_ckpts[-1])
    base_args += ["--resume-checkpoint", _last_ckpt]
    print(f"In-session resume from: {_last_ckpt}")
elif RESUME:
    print("RESUME_CHECKPOINTS=True but no local checkpoints found -- starting fresh")
    RESUME = False

# --- Multi-GPU: use torchrun when 2+ GPUs are available ---
n_gpus = _torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

if n_gpus > 1:
    import shutil as _shutil
    _torchrun = _shutil.which("torchrun") or "torchrun"
    cmd = [_torchrun, f"--nproc_per_node={n_gpus}", "--master_port=29500"] + base_args
    print(f"Launching DDP on {n_gpus} x T4")
else:
    cmd = [sys.executable] + base_args
    print("Single-GPU launch")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False)
print("Training exit code:", result.returncode)


Running: /usr/bin/python3 scripts/train.py --config configs/kaggle_config.yaml


INFO:__main__:✓ cuDNN autotuning enabled (first epoch may be slightly slower)
INFO:__main__:Mode: finetune
INFO:__main__:Config: configs/kaggle_config.yaml
INFO:__main__:Device: cuda
INFO:__main__:✓ Mixed precision (AMP) enabled (1.5-2× speedup expected)
INFO:__main__:
INFO:__main__:Starting Supervised Fine-tuning
INFO:__main__:============================================================
INFO:__main__:Creating dataloaders...
INFO:src.data.dataset:Loading graph from data/processed/biomedical_graph.pt
INFO:src.data.dataset:Graph loaded: gene=5363, disease=16841, phenotype=11794, ('gene', 'associated_with', 'disease')=9741610, ('disease', 'rev_associated_with', 'gene')=9741610
INFO:src.data.dataset:Loading gene-disease edges from data/processed/hpo_gene_disease_edges.csv
INFO:src.data.dataset:Vocabulary: 5251 genes, 12714 diseases
INFO:src.data.dataset:Loaded 1170143 edges ({'HPO_phenotype_bridge': 1170143})
INFO:src.data.dataset:Split sizes: train=936114, val=117014, test=117015
INFO:__m

## 10. Persist Checkpoints as Kaggle Output
Kaggle saves everything in `/kaggle/working/` as output after the session.  
Add those outputs as a new **Kaggle Dataset** so you can reload them next session.

In [ ]:
import shutil

OUTPUT_DIR = Path('/kaggle/working/saved_checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

src = Path('checkpoints/promptgfm_film')
for f in src.glob('*'):
    shutil.copy2(f, OUTPUT_DIR / f.name)

print(f'✅ Checkpoints copied to {OUTPUT_DIR}')
print('Files saved:')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 11. Quick Evaluation

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/evaluate.py',
     '--config', 'configs/kaggle_config.yaml',
     '--checkpoint', 'checkpoints/promptgfm_film/best_model.pt'],
    capture_output=False
)
print('Evaluation exit code:', result.returncode)